# Проект: предсказание стоимости жилья

В проекте мы должны обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. На основе данных нужно предсказать медианную стоимость дома в жилом массиве. Обучим модель и сделайте предсказания на тестовой выборке. Для оценки качества модели используйте метрики RMSE, MAE и R2.

<b>Цель исследования:</b> провести анализ данных для разработки модели, которая будет предсказывать медианную стоимость дома

<b>Основные задачи исследования:</b> провести предобработку данных, исследовательский анализ данных, корреляционный анализ данных и обучить модель

<b>Исходные данные:</b> Нам дан датасет с данными о домах в Калифорнии, вот с такими признаками:

- `longitude` — широта;
- `latitude` — долгота;
- `housing_median_age` — медианный возраст жителей жилого массива;
- `total_rooms` — общее количество комнат в домах жилого массива;
- `total_bedrooms` — общее количество спален в домах жилого массива;
- `population` — количество человек, которые проживают в жилом массиве;
- `households` — количество домовладений в жилом массиве;
- `median_income` — медианный доход жителей жилого массива;
- `median_house_value` — медианная стоимость дома в жилом массиве;
- `ocean_proximity` — близость к океану.

## Загрузка данных

Подгрузим используемые библиотеки

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

from pyspark.ml.feature import StringIndexer,VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import OneHotEncoder, Imputer
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

import pandas as pd
import numpy as np
import pyspark
import pyspark.sql.functions as F

Инициализируем локальную spark-сессию 

In [2]:
spark = SparkSession.builder \
                    .master("local") \
                    .appName("Predict_house_price") \
                    .getOrCreate()

Считаем наши данные и отобразим их, чтобы проверить адекватно ли они считались

In [3]:
df_main = spark.read.csv('/datasets/housing.csv',header=True,inferSchema=True)

In [4]:
df_main.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

Выведем названия столбцов и типы данных, которые содержатся в них

In [5]:
df_main.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



## Предобработка данных

Найдем пропуски в столбцах

In [6]:
columns = df_main.columns

for column in columns:
    check_col = F.isnan(F.col(column)) | F.isnull(F.col(column))
    print(column, df_main.filter(check_col).count())

longitude 0
latitude 0
housing_median_age 0
total_rooms 0
total_bedrooms 207
population 0
households 0
median_income 0
median_house_value 0
ocean_proximity 0


Ага, 207 прпопусков в столбце с суммарным количеством спален в домах, мы их заполним на этапе пайплайна в разделе с обучением модели

Теперь посмотрим, что там с явными дубликатами

In [7]:
print(df_main.count())

20640


In [8]:
df_main = df_main.dropDuplicates()
print(df_main.count())

20640


Явных дубликатов не было обнаружено, что там с неявными

In [9]:
display(df_main.groupBy('ocean_proximity').count().toPandas())

,ocean_proximity,count
0,ISLAND,5
1,NEAR OCEAN,2658
2,NEAR BAY,2290
3,<1H OCEAN,9136
4,INLAND,6551


Неявные дубликаты не были обнаружены

## Исследовательский анализ данных

Вызовем метод describe, чтобы провести хотя бы какой то базовый EDA

In [10]:
display(df_main.describe().toPandas())

,summary,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,count,20640,20640,20640,20640,20433,20640,20640,20640,20640,20640
1,mean,-119.56970445736432,35.6318614341085,28.639486434108527,2635.7630813953488,537.8705525375618,1425.4767441860465,499.5396802325581,3.8706710029069766,206855.81690891474,None
2,stddev,2.0035317235025882,2.135952397457116,12.585557612111657,2181.6152515827944,421.3850700740325,1132.4621217653407,382.32975283161085,1.8998217179452688,115395.61587441394,None
3,min,-124.35,32.54,1.0,2.0,1.0,3.0,1.0,0.4999,14999.0,<1H OCEAN
4,max,-114.31,41.95,52.0,39320.0,6445.0,35682.0,6082.0,15.0001,500001.0,NEAR OCEAN


Признаки широты и долготы имеют маленький разброс значений из за того, что условие задания диктует нам, что все дома находятся в Калифорнии. Можно предположить, что признак медианного возраста жителей распределен равномерно, так как имеет разброс значений от 1 до 52 и имеет медианное значение, которое располагается посередине этого разброса. Число комнат, спален, популяция и количество домовладений имеют огромный разброс и имеют при этом медиану более близкую к минимальному значению из чего можем предположить, что их распределение имеет выраженную ассиметрию. Примерно то же самое можно сказать и про медианный доход и стоимость дома, хотя у них медиана в соотношении находится подальше от минимального значения по сравнению с вышеуказанными 4-мя признаками

In [11]:
display(df_main.groupBy('ocean_proximity').count().toPandas())

,ocean_proximity,count
0,ISLAND,5
1,NEAR OCEAN,2658
2,NEAR BAY,2290
3,<1H OCEAN,9136
4,INLAND,6551


Что касается нашей категориальной переменной с близостью к океану, можно заметить, что подавляющее большинство живет в менее чем одном часе езды от океана, либо наоборот подальше от него, хотя также значительная часть людей живет возле залива или в возле океана

## Обучение моделей

Напишем пайплан для обработки данных и обучим модель на первом наборе

In [12]:
num_first = ['longitude',
             'latitude',
             'housing_median_age',
             'total_rooms',
             'total_bedrooms',
             'population',
             'households',
             'median_income']

num_second = ['housing_median_age',
             'total_rooms',
             'total_bedrooms',
             'population',
             'households',
             'median_income']

In [13]:
imputer = Imputer(inputCols=num_first, outputCols=[n+'_filled1' for n in num_first], strategy='mean')

indexer = StringIndexer(inputCols =['ocean_proximity'], outputCols=['cat_idx']) 

encoder = OneHotEncoder(inputCols =['cat_idx'],outputCols = ['cat_ohe'])

categorical_assembler = VectorAssembler(inputCols=['cat_ohe'], outputCol="categorical_features")

num_assembler = VectorAssembler(inputCols=[n+'_filled1' for n in num_first], outputCol="num_features1")

scaler = StandardScaler(inputCol="num_features1", outputCol="num_scaled_features1")

all_features = ['categorical_features','num_scaled_features1']

final_assembler = VectorAssembler(inputCols=all_features, outputCol="features1") 

lr = LinearRegression(featuresCol="features1", labelCol="median_house_value", predictionCol="prediction")

pipeline = Pipeline(stages=[imputer, indexer, encoder, categorical_assembler, num_assembler, scaler, final_assembler, lr])

(trainingData, testData) = df_main.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(trainingData)

predictions = model.transform(testData)

25/10/08 22:15:36 WARN Instrumentation: [0d900d80] regParam is zero, which might cause numerical instability and overfitting.
25/10/08 22:15:37 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/10/08 22:15:37 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
25/10/08 22:15:43 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
25/10/08 22:15:43 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


Посмотрим, что по метрикам

In [14]:
label_col = 'median_house_value'

evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")
r2 = evaluator.evaluate(predictions)

evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae")
mae = evaluator.evaluate(predictions)

print('rmse -',rmse)
print('r2 -',r2)
print('mae -',mae)

rmse - 65928.86152375725
r2 - 0.6701159132213483
mae - 48568.59041069315


Довольно таки плохо, особенно r2...

Посморим как себя покажет второй набор данных, в нем мы решили убрать широту и долготу, так как координаты не влияют напрямую на стоимость дома

In [15]:
imputer = Imputer(inputCols=num_second, outputCols=[n+'_filled2' for n in num_second], strategy='mean')

indexer = StringIndexer(inputCols =['ocean_proximity'], outputCols=['cat_idx']) 

encoder = OneHotEncoder(inputCols =['cat_idx'],outputCols = ['cat_ohe'])

categorical_assembler = VectorAssembler(inputCols=['cat_ohe'], outputCol="categorical_features")

num_assembler = VectorAssembler(inputCols=[n+'_filled2' for n in num_second], outputCol="num_features2")

scaler = StandardScaler(inputCol="num_features2", outputCol="num_scaled_features2")

all_features = ['categorical_features','num_scaled_features2']

final_assembler = VectorAssembler(inputCols=all_features, outputCol="features2") 

lr = LinearRegression(featuresCol="features2", labelCol="median_house_value", predictionCol="prediction")

pipeline = Pipeline(stages=[imputer, indexer, encoder, categorical_assembler, num_assembler, scaler, final_assembler, lr])

(trainingData, testData) = df_main.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(trainingData)

predictions = model.transform(testData)

25/10/08 22:16:19 WARN Instrumentation: [d1e9f8f9] regParam is zero, which might cause numerical instability and overfitting.


In [16]:
evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")
r2 = evaluator.evaluate(predictions)

evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae")
mae = evaluator.evaluate(predictions)

print('rmse -',rmse)
print('r2 -',r2)
print('mae -',mae)

rmse - 67075.14825763916
r2 - 0.6585449846117042
mae - 49365.53789151196


Стало еще хуже...

In [17]:
spark.stop()

## Анализ результатов

На первом этапе мы подгрузили наши данные, считались они без каких либо ошибок

На этапе предобработки данных мы заполнили пропуски модой, не нашли явных дубликатов и вычистили неявные в категориальном столбце

На этапе исследовательского анализа данных мы узнали, что большинство распределений числовых признаков имеют ассиметрию и медиану в близости к минимальному значению. А насчет категориального столбца - было выявлено, что люди предпочитают либо жить на в отдалении от океана, либо в часе от него.

Далее мы написали пайплайн для обучения и подготовки данных и обучили две модели линейной регрессии на двух разных наборах признаков. На первом наборе метрики получились довольно паршивые. На втором же наборе, где мы убрали признаки в связи с фактической неважностью для таргетной переменной, стало еще хуже

Исходя из результатов исследования можно предположить, что модели не хватает признаков, чтобы хорошо обучиться, стоит задуматься, требуется ли ML в данном кейсе